In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.5045, 10: 0.5214999999999999, 20: 0.535, 30: 0.5485000000000002, 40: 0.5589999999999999, 50: 0.5535000000000001, 60: 0.5590000000000002, 70: 0.5630000000000001, 80: 0.5745000000000001, 90: 0.5744999999999997, 100: 0.5834999999999997, 110: 0.5945, 120: 0.5914999999999997, 130: 0.6035000000000001, 140: 0.6129999999999999, 150: 0.6155000000000002, 160: 0.6164999999999997, 170: 0.6289999999999999, 180: 0.6279999999999999, 190: 0.6339999999999999, 200: 0.6439999999999999, 210: 0.6569999999999999, 220: 0.6685000000000001, 230: 0.6715, 240: 0.666, 250: 0.6625, 260: 0.6704999999999999, 270: 0.673, 280: 0.6825000000000001, 290: 0.6923076923076924, 300: 0.6948717948717947}
{0: 0.0036297500000000006, 10: 0.004747749999999999, 20: 0.0030749999999999996, 30: 0.00391775, 40: 0.003659, 50: 0.00312775, 60: 0.0038990000000000006, 70: 0.004951, 80: 0.00401975, 90: 0.0038597499999999995, 100: 0.0036977499999999988, 110: 0.0036997499999999995, 120: 0.0037577499999999994, 130: 0.0036577499999999995, 